# FT09 — baseline evaluation
Evaluate Qwen3.5-4B before FT09 fine-tuning, directly in a personal university GPU session using Unsloth.
Start with the three-turn Level 1 check below. No training runs in this notebook.


## Dependencies
Run once in your GPU session. If anything was installed or upgraded, **restart the kernel**, then continue below this cell.
Use the same installed versions in the team's sessions; `config.json` records them automatically.


In [ ]:
%pip install -q "arc-agi==0.9.9" numpy pillow matplotlib pyarrow
%pip install -q -U unsloth unsloth_zoo "transformers>=5,<6" accelerate bitsandbytes torchvision


In [ ]:
from pathlib import Path
import sys

# Open this notebook from the project folder containing ft09_shared.py.
PROJECT_DIR = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "ft09_shared.py").is_file()),
    None,
)
if PROJECT_DIR is None:
    raise RuntimeError("Open the notebook from the FT09 project folder.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import ft09_shared as ft
ft.configure_project(PROJECT_DIR)
print("Project:", PROJECT_DIR)


## 1. Settings
Leave `PHASE = "connection_check"` for the first check: Level 1, one attempt, three turns.
Run this check in two team sessions before starting the full evaluation. Check that clicks, raw replies,
token counts and summaries are saved, and that replies fit within the output budget. Then freeze the code/settings and tag `eval-v1`.

For the official runs, set `PHASE = "baseline"` and choose your assigned `LEVELS`.
The code automatically uses five attempts per level and the full agreed limits, all at temperature 1.0.
BF16 is the default; `LOAD_IN_4BIT = True` selects a separate 4-bit baseline.
Use the same model, precision, prompt and generation settings after fine-tuning.


In [ ]:
import getpass

MODEL = "unsloth/Qwen3.5-4B"
LOAD_IN_4BIT = False
PHASE = "connection_check"       # Change to "baseline" after the checks.
LEVELS = [1]                     # Baseline: your assigned levels from 1–5.
MAX_NEW_TOKENS = 1024
MAX_SEQ_LENGTH = 8192
TEMPERATURE = 1.0
SEED = 1234
SESSION = getpass.getuser()       # Distinguishes team members' output folders.
RUN_NAME = f"{PHASE}_01"          # Keep unchanged when resuming this batch.
RESUME = True
PROMPT = ft.GAME_PROMPT

if PHASE not in {"connection_check", "baseline", "heldout"}:
    raise ValueError("Choose connection_check, baseline or heldout.")
if not LEVELS or len(set(LEVELS)) != len(LEVELS):
    raise ValueError("Choose a nonempty list of different levels.")
if PHASE == "connection_check" and LEVELS != [1]:
    raise ValueError("Use Level 1 for the connection check.")
if PHASE != "heldout" and any(level not in range(1, 6) for level in LEVELS):
    raise ValueError("Keep Level 6 outside development and the levels 1–5 baseline.")
if PHASE == "heldout" and LEVELS != [6]:
    raise ValueError("The separate held-out batch must use LEVELS = [6].")
N_ATTEMPTS = 1 if PHASE == "connection_check" else 5
MAX_STEPS = 3 if PHASE == "connection_check" else None
SHOW_FRAMES = PHASE == "connection_check"


## 2. Load the model once
This cell downloads/loads the model on the selected GPU. Rerun after changing model or loading settings.
Extended thinking is disabled; the short `REASONING` field is still requested. All generated tokens are counted.


In [ ]:
from ft09_models import LocalQwenBackend

if "backend" in globals():
    backend.close()
backend = LocalQwenBackend(
    MODEL, load_in_4bit=LOAD_IN_4BIT,
    max_new_tokens=MAX_NEW_TOKENS, max_seq_length=MAX_SEQ_LENGTH,
)
print(backend.info())


## 3. Save settings and select the output folder
Results go to `eval/runs/<model>/<precision>/<session>/<run_name>/`.
The folder contains `config.json`, `system_prompt.txt`, raw request logs, turn records and summaries.
Keep the same session/run name to resume. Completed attempts are skipped. Partial attempts are restored
by replaying saved clicks locally; no model requests are needed for that replay.
A resumed GPU sampling sequence may differ from an uninterrupted run.

Changed settings/code/package versions require a new run name. Use the same frozen settings across the team.


In [ ]:
if (backend.model_name != MODEL or backend.max_new_tokens != MAX_NEW_TOKENS
        or backend.load_in_4bit != LOAD_IN_4BIT
        or backend.max_seq_length != MAX_SEQ_LENGTH):
    raise RuntimeError("Reload the model after changing the loading settings.")
RUN_DIR = ft.evaluation_run_dir(MODEL, LOAD_IN_4BIT, SESSION, RUN_NAME)
limits = {
    level: ft.step_limit(level, ft.get_baseline_actions())
    if MAX_STEPS is None else MAX_STEPS
    for level in LEVELS
}
ft.save_run_config(
    RUN_DIR, backend, PROMPT, resume=RESUME, phase=PHASE,
    levels=LEVELS, attempts=N_ATTEMPTS, temperature=TEMPERATURE,
    step_limits=limits, seed=SEED, session=SESSION,
)
backend.log_path = RUN_DIR / "requests.jsonl"
print("Output:", RUN_DIR)
print("Attempts:", N_ATTEMPTS, "| temperature:", TEMPERATURE, "| limits:", limits)


## 4. Run the selected levels
Each new attempt starts with a fresh board and empty memory. Every turn is saved.
Invalid/truncated replies and timeouts are counted; up to two retries are allowed per turn.
Technical failures save partial results and allow the next attempt to start. Interrupting the cell saves and stops.
Rerun this cell to resume unfinished attempts. Limits count decision turns, including invalid turns; executed clicks are counted separately.


In [ ]:
all_records = []
for level in LEVELS:
    all_records.extend(ft.run_level(
        backend, level, RUN_DIR,
        n_attempts=N_ATTEMPTS, temperature=TEMPERATURE,
        max_steps=MAX_STEPS, prompt=PROMPT, show_frames=SHOW_FRAMES,
        resume=RESUME, seed=SEED,
    ))


## 5. Results
Creates per-attempt `results.csv` and per-level `summary.csv`, including success, clicks, tokens, invalid replies and technical failures.
Timeouts are reported separately (normally zero with local inference).
Inspect technical failures before reporting game performance; they are not evidence that the model failed to solve the puzzle.


In [ ]:
attempt_results, level_summary = ft.summarise_evaluation(RUN_DIR)


## 6. Inspect a saved attempt


In [ ]:
FILE = next(iter(sorted(RUN_DIR.glob("ft09_level_*_attempt_*.parquet"))), None)
if FILE is not None:
    ft.review_steps(FILE, first_step=1, last_step=3)


## Held-out Level 6
After freezing the configuration, use `PHASE = "heldout"`, `LEVELS = [6]` and a separate `RUN_NAME`.
Report those results separately. Keep Level 6 out of development and training.

After changing shared Python files, restart the kernel. Do not reinstall/upgrade packages between official runs.
Copy your full run folder when sharing results, including configuration and logs.
